<a href="https://colab.research.google.com/github/21centjoe/Aleph-NELOS-1D-/blob/main/MYOSYNTHETICQUANTUMSOLITONGENERATOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Quantum Soliton Processing System

This implementation integrates the Cirq library to simulate quantum state preparation, specifically modeled to generate a soliton waveform. The resulting data is processed through a chromatic 3D filter and saved as a high-resolution bitmap.

The mathematical foundation for the wave packet follows:


$$\psi(x, t) = \text{sech}(252 \cdot (x - vt))$$

In [ ]:
# Install necessary libraries.
# Let pip handle dependency resolution for numpy, scipy, and pandas
!pip install cirq

### Quantum Soliton Frame Generation

This section generates the individual frames for the quantum soliton animation. These frames will then be compiled into a video using `ffmpeg`.

In [ ]:
import cirq
import numpy as np
from PIL import Image, ImageDraw
import os

def generate_quantum_soliton_batch(iterations=10):
    # Setup parameters
    size = 4096
    output_dir = "quantum_soliton_outputs"
    num_qubits = 12

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Load Cirq simulator
    qubits = cirq.LineQubit.range(num_qubits)
    simulator = cirq.Simulator()

    axis = np.linspace(-1, 1, size)
    x, y = np.meshgrid(axis, axis)

    for i in range(iterations):
        # 1. Quantum Circuit Construction
        circuit = cirq.Circuit()
        # Scale factor incorporates 252 for the soliton peak
        shift = i * 0.05

        for j, q in enumerate(qubits):
            # Heavy lifting: Creating the probability distribution
            angle = np.pi * np.exp(-((j - num_qubits/2 + shift)**2))
            circuit.append(cirq.ry(angle)(q))

        # 2. Execute Simulation
        result = simulator.simulate(circuit)
        amplitudes = np.abs(result.final_state_vector)

        # 3. Soliton Wave Mapping (Spatial Grid)
        # Mapping the [2m] temporal span into spatial coordinates
        mag = np.interp(np.sqrt(x**2 + y**2), np.linspace(0, 1, len(amplitudes)), amplitudes)

        # 4. Chromatic 3D Separation (Red, Green, Yellow/Blue)
        # Offset channels to create the printed 3D effect
        r_chan = (np.sin(252 * (mag + 0.002)) > 0).astype(np.uint8) * 255
        g_chan = (np.sin(252 * mag) > 0).astype(np.uint8) * 255
        b_chan = (np.sin(252 * (mag - 0.002)) > 0).astype(np.uint8) * 255

        # Combine into RGB stack
        rgb_stack = np.stack([r_chan, g_chan, b_chan], axis=-1)
        img = Image.fromarray(rgb_stack, mode='RGB')

        # 5. Statement Overlay
        draw = ImageDraw.Draw(img)
        statement = "my first Quantum inspired bitmap"

        # Position box at the base
        box_coords = [100, size - 300, 2400, size - 150]
        draw.rectangle(box_coords, fill=(0, 0, 0))
        draw.text((150, size - 280), statement, fill=(255, 255, 255))

        # 6. Save File
        filename = f"soliton_quantum_{i+1}.bmp"
        img.save(os.path.join(output_dir, filename))
        print(f"Generated: {filename}")

if __name__ == "__main__":
    generate_quantum_soliton_batch(10)

In [ ]:
import os

# Set the output directory for images
output_dir = "quantum_soliton_outputs"

# List all generated image files
image_files = sorted([os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.bmp')])

# Check if any images were found
if not image_files:
    print(f"No BMP files found in {output_dir}. Please run the image generation code first.")
else:
    print(f"Found {len(image_files)} image files.")

Next, we will install `ffmpeg` which is a powerful command-line tool used for converting multimedia files. It's often used for tasks like creating videos from image sequences, compressing videos, and changing video formats.

In [ ]:
# Install ffmpeg, a command-line tool for video and audio processing.
# The `-y` flag automatically answers yes to prompts during installation.
!apt-get install -y ffmpeg

Now, we'll use `ffmpeg` to create the video from the sequence of generated `.bmp` images. We'll set a frame rate of 10 frames per second.

In [ ]:
import os

# Set the output directory for images
output_dir = "quantum_soliton_outputs"

# List all generated image files
image_files = sorted([os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.bmp')])

# Check if any images were found (This block is for informational output, not strictly needed for functionality here)
if not image_files:
    print(f"No BMP files found in {output_dir}. Please run the image generation code first.")
else:
    print(f"Found {len(image_files)} image files.")

# Define the output video filename
video_filename = "quantum_soliton_animation.mp4"

# Construct the ffmpeg command.
# -r 10: Sets the input frame rate to 10 frames per second.
# -i: Specifies the input file pattern. 'quantum_soliton_outputs/soliton_quantum_%d.bmp' tells ffmpeg to look for files named soliton_quantum_1.bmp, soliton_quantum_2.bmp, etc.
# -vf "scale=1920:-1": Video filter to scale the output to a width of 1920 pixels, maintaining aspect ratio.
# -c:v libx264: Specifies the video codec to use (H.264).
# -pix_fmt yuv420p: Sets the pixel format, ensuring compatibility with most players.
# -o: Specifies the output filename.

# Note: The %d in the input path expects sequentially numbered files starting from 1.
# If the filenames are not sequential (e.g., they skip numbers or start from 0), this command might need adjustment.
# For this specific notebook, files are named `soliton_quantum_1.bmp`, `soliton_quantum_2.bmp`, etc., which works well with `%d`.

# Get the first image number from the sorted list to use with ffmpeg's start_number option
if image_files:
    start_number = int(os.path.basename(image_files[0]).replace('soliton_quantum_', '').replace('.bmp', ''))

    # Generate a temporary text file with the list of images for ffmpeg concatenation
    with open('image_list.txt', 'w') as f:
        for img_file in image_files:
            f.write(f"file '{img_file}'\n")

    # Use ffmpeg concat demuxer for robust image sequence processing
    # -f concat: Use the concat demuxer.
    # -safe 0: Allow unconventional filenames (needed for file list).
    # -i image_list.txt: Input is the list of image files.
    # -framerate 10: Set output framerate
    # -vsync vfr: Variable framerate to match input if needed.
    # -pix_fmt yuv420p: Pixel format for broad compatibility.
    # -c:v libx264: H.264 video codec.
    # -crf 18: Constant Rate Factor for video quality (lower is better quality, larger file size). Lowered from 23 to 18 for higher quality.
    # -preset slow: Use a slower encoding preset for better compression and quality.
    # -threads 0: Use all available CPU cores.
    # -y: Overwrite output file without asking.
    !ffmpeg -r 10 -f concat -safe 0 -i image_list.txt -vsync vfr -pix_fmt yuv420p -c:v libx264 -crf 18 -preset slow -threads 0 -y {video_filename}

    print(f"Video '{video_filename}' created successfully!")
    print(f"You can download it from /content/{video_filename}")
else:
    print("No images to create a video from.")

In [ ]:
from IPython.display import Video

# Path to the generated video file
video_path = '/content/quantum_soliton_animation.mp4'

# Display the video in the notebook
Video(video_path, embed=True)

### Dynamic Fractal and Geometric Visualization with Music

As outlined, we'll proceed by first generating animated Mandelbrot fractals with metallic color schemes. This will be the base upon which we'll add geometric shapes, kaleidoscope effects, and fluid-like distortions. Finally, we'll combine the visual output with your specified music.

The following cell (`523fa31d`) contains the code to generate the Mandelbrot fractal frames. Please run this cell to start generating the frames. This process may take a few minutes.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
import os

def create_metallic_color_palette(num_colors):
    """Generates a metallic-looking color palette."""
    colors = []
    for i in range(num_colors):
        # A sinusoidal function to create shifting hues
        hue = (i / num_colors) * 0.3 + 0.6  # Shift hue towards blues/purples
        saturation = 0.7 + 0.3 * np.sin(np.pi * (i / num_colors))
        value = 0.5 + 0.5 * np.cos(np.pi * (i / num_colors) + np.pi/2)

        # Convert HSV to RGB for better control over metallic look
        # This is a simplified HSV to RGB conversion for effect
        r = int(255 * (value * (1 + saturation * np.cos(hue * 2 * np.pi)))) % 256
        g = int(255 * (value * (1 + saturation * np.cos(hue * 2 * np.pi + np.pi/2)))) % 256
        b = int(255 * (value * (1 + saturation * np.cos(hue * 2 * np.pi + np.pi)))) % 256

        colors.append((r, g, b))
    return colors

def generate_mandelbrot_frame(width, height, x_center, y_center, zoom, max_iter, palette):
    """Generates a single frame of a Mandelbrot set with a given palette."""
    img = Image.new('RGB', (width, height))
    pixels = img.load()

    x_min = x_center - (1 / zoom)
    x_max = x_center + (1 / zoom)
    y_min = y_center - (1 / zoom)
    y_max = y_center + (1 / zoom)

    for row in range(height):
        for col in range(width):
            # Scale pixel coordinates to complex plane coordinates
            cx = x_min + (col / width) * (x_max - x_min)
            cy = y_min + (row / height) * (y_max - y_min)

            c = complex(cx, cy)
            z = 0 + 0j
            num_iter = 0

            while abs(z) < 2 and num_iter < max_iter:
                z = z * z + c
                num_iter += 1

            # Map iteration count to color from the palette
            if num_iter == max_iter:
                pixels[col, row] = (0, 0, 0) # Black for points inside the set
            else:
                pixels[col, row] = palette[num_iter % len(palette)]
    return img

def draw_shapes_on_frame(img, frame_idx, width, height, num_frames):
    """Draws animated geometric shapes on the image."""
    draw = ImageDraw.Draw(img)

    # Example 1: Rotating rectangle
    angle = (frame_idx / num_frames) * 360 # Rotate 360 degrees over the animation
    center_x, center_y = width // 2, height // 2
    rect_width, rect_height = width // 4, height // 4

    # Calculate rotated corners
    points = []
    for dx, dy in [(-rect_width/2, -rect_height/2), (rect_width/2, -rect_height/2),
                   (rect_width/2, rect_height/2), (-rect_width/2, rect_height/2)]:
        x = dx * np.cos(np.deg2rad(angle)) - dy * np.sin(np.deg2rad(angle))
        y = dx * np.sin(np.deg2rad(angle)) + dy * np.cos(np.deg2rad(angle))
        points.append((center_x + x, center_y + y))
    draw.polygon(points, outline=(255, 255, 0), width=3) # Yellow outline

    # Example 2: Expanding circle
    radius_max = min(width, height) // 3
    current_radius = (frame_idx / num_frames) * radius_max
    draw.ellipse((center_x - current_radius, center_y - current_radius,
                  center_x + current_radius, center_y + current_radius),
                 outline=(0, 255, 255), width=2) # Cyan outline

    # Example 3: Small square following a path
    path_radius = min(width, height) // 5
    path_angle = (frame_idx / num_frames) * 720 # Twice the rotation
    square_size = 20

    square_center_x = center_x + path_radius * np.cos(np.deg2rad(path_angle))
    square_center_y = center_y + path_radius * np.sin(np.deg2rad(path_angle))

    draw.rectangle((square_center_x - square_size/2, square_center_y - square_size/2,
                    square_center_x + square_size/2, square_center_y + square_size/2),
                   fill=(255, 0, 255)) # Magenta fill



# --- Parameters for Mandelbrot animation ---
output_dir_fractal = "fractal_animation_frames"
if not os.path.exists(output_dir_fractal):
    os.makedirs(output_dir_fractal)

image_width = 1024
image_height = 1024
num_frames = 50 # Changed from 100 to 50 for faster demonstration
max_iterations = 100

# Generate a metallic color palette
metallic_palette = create_metallic_color_palette(max_iterations)

# Animation parameters (zoom and center movement)
start_zoom = 0.5
end_zoom = 1000

start_x_center = -0.75
end_x_center = -0.7436

start_y_center = 0.0
end_y_center = 0.131

print(f"Generating {num_frames} fractal frames...")
for i in range(num_frames):
    t = i / (num_frames - 1) # Normalized time from 0 to 1

    current_zoom = start_zoom * (end_zoom / start_zoom)**t
    current_x_center = start_x_center + (end_x_center - start_x_center) * t
    current_y_center = start_y_center + (end_y_center - start_y_center) * t

    frame_img = generate_mandelbrot_frame(
        image_width, image_height,
        current_x_center, current_y_center,
        current_zoom, max_iterations, metallic_palette
    )

    # Overlay geometric shapes on the frame
    draw_shapes_on_frame(frame_img, i, image_width, image_height, num_frames)

    # Save the frame
    frame_filename = os.path.join(output_dir_fractal, f"mandelbrot_frame_{i:04d}.png")
    frame_img.save(frame_filename)
    if (i + 1) % 10 == 0 or i == num_frames - 1:
        print(f"Generated frame {i+1}/{num_frames}")

print("Mandelbrot fractal frames generation complete.")

Generating 50 fractal frames...
Generated frame 10/50


In [ ]:
import os

video_filename_fractal = "mandelbrot_animation.mp4"
output_dir_fractal = "fractal_animation_frames"

# Ensure ffmpeg is installed
!apt-get install -y ffmpeg

# Check if there are frames to process
if os.path.exists(output_dir_fractal) and len(os.listdir(output_dir_fractal)) > 0:
    # -framerate 20: Sets the input frame rate to 20 frames per second.
    # -i: Pattern for input files
    # -c:v libx264: H.264 video codec
    # -pix_fmt yuv420p: High compatibility pixel format
    # -y: Overwrite output file
    ffmpeg_command = (
        f"ffmpeg -framerate 20 -i {output_dir_fractal}/mandelbrot_frame_%04d.png "
        f"-c:v libx264 -pix_fmt yuv420p -crf 23 -y {video_filename_fractal}"
    )
    print(f"Executing: {ffmpeg_command}")
    !{ffmpeg_command}
    print(f"\nVideo '{video_filename_fractal}' created successfully!")
else:
    print("Error: No fractal frames found. Please run the generation cell first.")

Now that we have generated the fractal frames, we can combine them into a video. We'll use `ffmpeg` for this. This step also serves as a checkpoint to ensure the image generation is working as expected before adding more complex effects.

In [ ]:
video_filename_fractal = "mandelbrot_animation.mp4"

# Ensure ffmpeg is installed
!apt-get install -y ffmpeg

# Use ffmpeg to create a video from the generated PNG frames
# -framerate 20: Sets the input frame rate to 20 frames per second.
# -i: Specifies the input file pattern. 'fractal_animation_frames/mandelbrot_frame_%04d.png'
#     tells ffmpeg to look for files named mandelbrot_frame_0000.png, mandelbrot_frame_0001.png, etc.
# -c:v libx264: Specifies the video codec to use (H.264).
# -pix_fmt yuv420p: Sets the pixel format for broad compatibility.
# -crf 23: Constant Rate Factor for video quality (lower is better, larger file size).
# -y: Overwrite output file without asking.

# Check if there are frames to process
if os.path.exists(output_dir_fractal) and len(os.listdir(output_dir_fractal)) > 0:
    ffmpeg_command = (
        f"ffmpeg -framerate 20 -i {output_dir_fractal}/mandelbrot_frame_%04d.png "
        f"-c:v libx264 -pix_fmt yuv420p -crf 23 -y {video_filename_fractal}"
    )
    !{ffmpeg_command}
    print(f"Video '{video_filename_fractal}' created successfully!")
    print(f"You can download it from /content/{video_filename_fractal}")
else:
    print("No fractal frames found to create a video.")


### Adding Music to the Video

To add music to our generated video, we'll use `ffmpeg` again. Please upload your desired music file to your Colab environment (e.g., in the `/content/` directory) or provide its path if it's already accessible. Then, specify the path in the `audio_file_path` variable below.

In [ ]:
import os
import shlex # Import shlex for safe shell quoting

# @title Define Audio File Path and Merge with Video

# IMPORTANT: Replace 'path/to/your/music.mp3' with the actual path to your audio file.
# For example: audio_file_path = '/content/my_cool_music.mp3'
audio_file_path = "" # @param {type:"string"}

input_video_filename = "mandelbrot_animation.mp4"
output_video_with_audio_filename = "final_mandelbrot_animation_with_music.mp4"

# Check if the input video exists
if not os.path.exists(input_video_filename):
    print(f"Error: Input video '{input_video_filename}' not found. Please ensure it was generated successfully.")
elif not os.path.exists(audio_file_path):
    print(f"Error: Audio file '{audio_file_path}' not found. Please upload your music file and update the 'audio_file_path' variable.")
else:
    # ffmpeg command to combine video and audio
    # -i: Specifies input files (video and audio)
    # -c:v copy: Copy the video stream without re-encoding (faster, preserves quality)
    # -c:a aac: Encode audio to AAC format (common and widely supported)
    # -b:a 192k: Set audio bitrate to 192 kbps
    # -shortest: Finish encoding when the shortest input stream ends (i.e., video or audio)
    # -y: Overwrite output file without asking
    ffmpeg_command_audio = (
        f"ffmpeg -i {shlex.quote(input_video_filename)} "
        f"-i {shlex.quote(audio_file_path)} "
        f"-c:v copy -c:a aac -b:a 192k -shortest -y "
        f"{shlex.quote(output_video_with_audio_filename)}"
    )
    print(f"Executing: {ffmpeg_command_audio}")
    !{ffmpeg_command_audio}

    print(f"\nFinal video with music '{output_video_with_audio_filename}' created successfully!")
    print(f"You can download it from /content/{output_video_with_audio_filename}")

In [ ]:
from IPython.display import Video

# Path to the generated video file with audio
video_path_with_audio = '/content/final_mandelbrot_animation_with_music.mp4'

# Display the video in the notebook
Video(video_path_with_audio, embed=True)

Once `final_mandelbrot_animation_with_music.mp4` has been generated by running cells `b1a1bf69` and `b561d552`, you can use the following `ffmpeg` command to trim its duration. Replace `START_TIME` (e.g., `00:00:10` for 10 seconds in) and `DURATION` (e.g., `00:00:30` for a 30-second clip) with your desired values.

### Verify Generated Frames

To confirm that the frames for both the Quantum Soliton and Mandelbrot animations have been successfully generated, you can list the files in their respective output directories.

In [ ]:
import os

print("Files in /content/ directory:")
for f in os.listdir('/content/') :
  print(f)

In [ ]:
import os

video_file = "mandelbrot_animation.mp4"

if os.path.exists(video_file):
    print(f"The file '{video_file}' exists in the current directory (/content/).")
else:
    print(f"The file '{video_file}' does NOT exist in the current directory (/content/).")

In [ ]:
import os

final_video = "final_mandelbrot_animation_with_music.mp4"

if os.path.exists(final_video):
    print(f"Success! '{final_video}' exists and is ready.")
else:
    print(f"The file '{final_video}' was not found. Please ensure you have uploaded the audio file and executed cell b561d552.")

In [ ]:
import os

# Verify Quantum Soliton frames
quantum_soliton_output_dir = "quantum_soliton_outputs"
print(f"Files in '{quantum_soliton_output_dir}':")
if os.path.exists(quantum_soliton_output_dir):
    for f in os.listdir(quantum_soliton_output_dir):
        print(f)
else:
    print(f"Directory '{quantum_soliton_output_dir}' not found.")

print("\n" + "="*50 + "\n")

# Verify Mandelbrot fractal frames
fractal_output_dir = "fractal_animation_frames"
print(f"Files in '{fractal_output_dir}':")
if os.path.exists(fractal_output_dir):
    for f in os.listdir(fractal_output_dir):
        print(f)
else:
    print(f"Directory '{fractal_output_dir}' not found.")

In [ ]:
import shlex
import os

input_video = "/content/final_mandelbrot_animation_with_music.mp4"
output_trimmed_video = "/content/trimmed_mandelbrot_animation.mp4"

# --- Customize these values ---
START_TIME = "00:00:05" # Start trimming from 5 seconds into the video (HH:MM:SS format)
DURATION = "00:00:15"  # Keep a 15-second duration from the start time
# -------------------------------

if os.path.exists(input_video):
    ffmpeg_trim_command = (
        f"ffmpeg -ss {shlex.quote(START_TIME)} "
        f"-i {shlex.quote(input_video)} "
        f"-t {shlex.quote(DURATION)} "
        f"-c copy "
        f"-y {shlex.quote(output_trimmed_video)}"
    )
    print(f"Executing: {ffmpeg_trim_command}")
    !{ffmpeg_trim_command}

    print(f"\nTrimmed video '{output_trimmed_video}' created successfully!")
    print(f"You can download it from /content/{output_trimmed_video}")
else:
    print(f"Error: Input video '{input_video}' not found. Please ensure it was generated first.")

